# dscraft.automl quickstart

This notebook demonstrates `dscraft.automl`'s signature capability: `compile()`, which takes a **fitted** `sklearn.pipeline.Pipeline` and fuses it into a single, portable `onnx.ModelProto` via `skl2onnx`.

We fit a `StandardScaler` + `LogisticRegression` pipeline on the classic Iris toy dataset, compile it to ONNX, run inference through `onnxruntime`, and cross-check that the ONNX graph's predictions match the original sklearn pipeline's own predictions.

This mirrors `packages/dscraft/examples/automl/compile_iris_example.py` (same dataset, same pipeline, same correctness checks) in notebook form.

Requires the `automl` and `automl-onnx` extras:
```bash
pip install -e "packages/dscraft[automl,automl-onnx]"
```

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Aliased to `dscraft_compile` because `compile` shadows the Python builtin
# of the same name (Ruff A004) -- this mirrors the `# noqa: A004` comment
# already present in `dscraft/automl/__init__.py` itself for this same
# intentional-but-shadowing public API name.
from dscraft.automl import CompileOptions, compile as dscraft_compile

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.3, random_state=42, stratify=iris.target
)

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)
pipeline.fit(X_train, y_train)

sklearn_accuracy = pipeline.score(X_test, y_test)
print(f"Fitted sklearn Pipeline test accuracy: {sklearn_accuracy:.4f}")

Fitted sklearn Pipeline test accuracy: 0.9111


## Compile the fitted pipeline to a single ONNX graph

In [2]:
import onnxruntime

onnx_model = dscraft_compile(pipeline, X_test, options=CompileOptions(zipmap=False))
print(f"Compiled ONNX graph with {len(onnx_model.graph.node)} nodes.")

session = onnxruntime.InferenceSession(
    onnx_model.SerializeToString(), providers=["CPUExecutionProvider"]
)
input_name = session.get_inputs()[0].name
X_test_f32 = X_test.astype(np.float32)
onnx_labels, onnx_proba = session.run(None, {input_name: X_test_f32})

Compiled ONNX graph with 3 nodes.


## Cross-check: ONNX predictions vs. the original sklearn pipeline

In [3]:
sklearn_labels = pipeline.predict(X_test)
sklearn_proba = pipeline.predict_proba(X_test)

labels_match = np.array_equal(np.asarray(onnx_labels).astype(sklearn_labels.dtype), sklearn_labels)
proba_close = np.allclose(np.asarray(onnx_proba), sklearn_proba, atol=1e-4)

print(f"ONNX predicted labels match sklearn exactly: {labels_match}")
print(f"ONNX predicted probabilities match sklearn within 1e-4: {proba_close}")

assert labels_match, "ONNX label predictions diverged from the sklearn pipeline."
assert proba_close, "ONNX predict_proba output diverged from the sklearn pipeline."

print("compile() correctness check passed: ONNX graph matches sklearn pipeline.")

ONNX predicted labels match sklearn exactly: True
ONNX predicted probabilities match sklearn within 1e-4: True
compile() correctness check passed: ONNX graph matches sklearn pipeline.


## Summary

This notebook fit a real sklearn `Pipeline` on the Iris dataset, compiled it into a single, portable ONNX graph via `dscraft.automl.compile()`, and confirmed the ONNX graph's predictions match the original sklearn pipeline's own predictions within tolerance.

See `packages/dscraft/README.md`'s `## dscraft.automl` section for more detail on `compile()`'s scope and design.